# PDFs Data Cleaning

In [2]:
import os
pdfs_path = r"../../datasets/training/"
for file in os.listdir(pdfs_path):
    print(file)

CAO - 1.pdf
CAO - 1._cleaned.txt
CAO - 2.pdf
CN - 1.pdf
CN - 2.pdf
CN - 3.pdf
DSA - 1.pdf
DSA - 2.pdf
DSA - 3.pdf
DSA - 4.pdf
OS - 1.pdf
OS - 1._cleaned.txt
OS - 2.pdf
OS - 3.pdf


## Image descriptions

### Extract Images from pdfs

In [50]:
def extract_images_from_pdf(pdf_path, output_folder, min_width=500, min_height=500):
    """
    Extract images from a PDF only if they meet minimum width and height.

    Parameters:
        pdf_path (str or Path): Path to PDF file.
        output_folder (str or Path): Folder to save images.
        min_width (int): Minimum width in pixels.
        min_height (int): Minimum height in pixels.
    """
    pdf_path = Path(pdf_path)
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)
    
    doc = fitz.open(pdf_path)
    image_count = 0

    for page_number, page in enumerate(doc, start=1):
        image_list = page.get_images(full=True)
        for img_index, img in enumerate(image_list, start=1):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            image_ext = base_image["ext"]
            width = base_image["width"]
            height = base_image["height"]

            # Skip small images
            if width < min_width or height < min_height:
                continue

            image_filename = output_folder / f"{pdf_path.stem}_page{page_number}_img{img_index}.{image_ext}"
            with open(image_filename, "wb") as img_file:
                img_file.write(image_bytes)
            
            image_count += 1

    doc.close()
    print(f"Extracted {image_count} images meeting size requirements from {pdf_path.name} into {output_folder}")

In [51]:
file_path = pdfs_path + "OS - 1.pdf" 
extract_images_from_pdf(file_path,"./images",250,250)

Extracted 43 images meeting size requirements from OS - 1.pdf into images


### Generate image descriptions

In [33]:
!pip install strip-markdown

/bin/bash: /home/sasank-v/anaconda3/envs/ml/lib/libtinfo.so.6: no version information available (required by /bin/bash)


In [40]:
import base64
from openai import OpenAI
from strip_markdown import strip_markdown

vision_client = OpenAI(base_url="",api_key="")
VISION_LLM_MODEL_NAME="Qwen/Qwen3-VL-8B-Instruct"

def get_image_description(image_bytes,prompt):
    image_base64 = base64.b64encode(image_bytes).decode("utf-8")
    response = vision_client.chat.completions.create(
        model=VISION_LLM_MODEL_NAME,
        temperature=0.1,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{image_base64}"
                        },
                    },
                ],
            }
        ],
    )
    choice = response.choices[0]
    desc = choice.message.content.strip()
    return strip_markdown(desc)

In [41]:
vision_prompt="""
You are an expert academic analyst specializing in technical diagrams, scientific figures, engineering block diagrams, flowcharts, and data visualizations.

Carefully analyze the provided image and produce a comprehensive, academically structured description suitable for inclusion in a research paper or technical documentation.

Follow these instructions strictly:

1. Identify the type of visual representation (e.g., block diagram, system architecture diagram, flowchart, circuit schematic, line graph, bar chart, scatter plot, etc.).

2. Transcribe all visible text exactly as it appears, including labels, legends, axis titles, units, annotations, and captions.

3. Describe the structural layout:

   * Relative positioning of components (top, bottom, left, right, center).
   * Hierarchical organization, if any.
   * Modular grouping of elements.

4. Describe each component in detail:

   * Name/label
   * Shape (rectangle, ellipse, diamond, arrow, etc.)
   * Function (if inferable from context)
   * Inputs and outputs

5. Describe relationships and connections:

   * Directionality of arrows
   * Data/control flow
   * Feedback loops
   * Bidirectional links
   * Conditional branches (if present)

6. If the image is a graph or chart:

   * Identify axes and units
   * Describe scale (linear, logarithmic)
   * Identify trends, peaks, inflection points, correlations
   * Compare relative magnitudes
   * Describe any statistical indicators (error bars, confidence intervals, regression lines)

7. Provide analytical interpretation:

   * What system or process is being represented?
   * What is the apparent objective of the diagram?
   * What conclusions can reasonably be inferred?
   * What assumptions seem implicit?

8. Avoid vague language.
   Do not say “the diagram shows something.”
   Be specific and technical.

9. If any element is unclear or ambiguous, explicitly state the uncertainty instead of guessing.

Structure your output in clearly labeled sections:

* Type of Visual
* Text Transcription
* Structural Layout
* Component Analysis
* Relationships and Flow
* Analytical Interpretation
* Notable Observations
* Uncertainties (if any)

Write in formal academic tone.
"""

In [48]:
image_path="./images/OS - 1_page45_img1.jpeg"
image_bytes=""
with open(image_path,"rb") as img:
    image_bytes = img.read()
get_image_description(image_bytes,vision_prompt)

'Type of Visual\nSystem architecture diagram illustrating a symmetric multiprocessing (SMP) configuration with two identical processor units and shared main memory.\nText Transcription\n- Top-left box: processor_0\n- Top-right box: processor_1\n- Left inner box: CPU_0\n- Right inner box: CPU_1\n- Left inner component: registers\n- Right inner component: registers\n- Left inner component: cache\n- Right inner component: cache\n- Bottom box: main memory\nStructural Layout\nThe diagram is horizontally symmetrical, with two identical processor modules positioned side-by-side at the top level. Each processor module is enclosed in a light cyan rectangular boundary. Within each processor module, a gray rectangular boundary contains two stacked white rectangular components: registers above cache. Below the two processor modules, a single light cyan rectangular block labeled main memory is centered. Black lines connect the cache component of each processor to the main memory block, forming a sy

## Extract text with image descriptions

In [54]:
def extract_text_with_image_descriptions(pdf_path, output_path, prompt, min_height=500, min_width=500):
    doc = fitz.open(pdf_path)
    full_text = ""

    for page in doc:
        blocks = page.get_text("dict")["blocks"]

        for block in blocks:

            # TEXT BLOCK
            if block["type"] == 0:
                for line in block.get("lines", []):
                    for span in line.get("spans", []):
                        full_text += span["text"]
                full_text += "\n"

            # IMAGE BLOCK
            elif block["type"] == 1:

                # Case 1: image data is directly inside block
                if "image" in block:
                    image_bytes = block["image"]
                    width = block.get("width", 0)
                    height = block.get("height", 0)

                # Case 2: image referenced by xref
                elif "xref" in block:
                    xref = block["xref"]
                    base_image = doc.extract_image(xref)
                    image_bytes = base_image["image"]
                    width = base_image["width"]
                    height = base_image["height"]

                else:
                    # Unknown structure — skip safely
                    continue

                # Filter small images
                if height < min_height or width < min_width:
                    continue

                description = get_image_description(image_bytes, prompt)

                full_text += "\n[Image Description Start]\n"
                full_text += description
                full_text += "\n[Image Description End]\n\n"

        full_text += "\n"
    doc.close()
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(full_text)

In [55]:
file_path = file_path = pdfs_path + "OS - 1.pdf" 
output_path = output_path = file_path[:-3] + "_full.txt"
extract_text_with_image_descriptions(file_path,output_path,vision_prompt,250,250)

# Remove unwanted content 
- index pages
- acknowledgements
- publisher boilerplate
- references section
- page headers/footers

In [3]:
!pip install pymupdf

/bin/bash: /home/sasank-v/anaconda3/envs/ml/lib/libtinfo.so.6: no version information available (required by /bin/bash)


In [8]:
import os
import re
import fitz  # PyMuPDF
from pathlib import Path

def extract_text_from_pdf(pdf_path):
    """
    Extract raw text from a PDF file using PyMuPDF.
    """
    doc = fitz.open(pdf_path)
    text = ""
    print(len(doc))
    for page in doc:
        page_text = page.get_text("text")
        text += page_text + "\n"
    doc.close()
    return text

def remove_headers_footers(text):
    lines = text.split("\n")
    cleaned_lines = []

    for line in lines:
        line_strip = line.strip()

        # Remove standalone page numbers
        if re.fullmatch(r"\d+", line_strip):
            continue

        cleaned_lines.append(line)

    return "\n".join(cleaned_lines)


def remove_unwanted_sections(text):
    """
    Remove entire sections ONLY if they appear as a heading.
    Assumes headings are standalone lines.
    """

    section_headings = [
        "index",
        "acknowledgment",
        "acknowledgements",
        "references",
        "bibliography"
    ]

    lines = text.split("\n")
    cleaned_lines = []
    skip = False

    for line in lines:
        line_strip = line.strip().lower()

        # If line is exactly a section heading, stop keeping content
        if line_strip in section_headings:
            skip = True
            continue

        if not skip:
            cleaned_lines.append(line)

    return "\n".join(cleaned_lines)


def fix_broken_lines(text):
    """
    Merge lines that are broken mid-sentence.
    """
    # Replace single newline between words with space
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)

    # Normalize excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text


def clean_text(raw_text):
    text = raw_text

    text = remove_headers_footers(text)
    text = remove_unwanted_sections(text)
    text = fix_broken_lines(text)

    # Normalize spaces
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def process_pdf(pdf_file_path, output_file_path=None):
    """
    Extract, clean, and optionally save a PDF.

    Parameters:
        pdf_file_path (str or Path)
        output_file_path (str or Path, optional)

    Returns:
        formatted_text (str)
    """

    pdf_file_path = Path(pdf_file_path)

    raw_text = extract_text_from_pdf(pdf_file_path)
    print("RAW LENGTH:", len(raw_text))
    
    cleaned_text = clean_text(raw_text)
    print("CLEAN LENGTH:", len(cleaned_text))


    subject_name = pdf_file_path.stem
    formatted_text = (
        f"\n\n===== SOURCE: {subject_name} =====\n\n"
        f"{cleaned_text}\n"
    )

    # Write formatted text if output path provided
    if output_file_path:
        output_file_path = Path(output_file_path)
        with open(output_file_path, "w", encoding="utf-8") as f:
            f.write(formatted_text)

    return formatted_text


def process_pdf_folder(input_folder, output_file, save_individual=False):
    """
    Process all PDFs in a folder and save cleaned content.

    Parameters:
        input_folder (str or Path)
        output_file (str or Path)
        save_individual (bool): Whether to save cleaned individual files.
    """

    input_folder = Path(input_folder)
    output_file = Path(output_file)

    all_cleaned_text = ""

    for pdf_file in sorted(input_folder.glob("*.pdf")):
        print(f"Processing: {pdf_file.name}")

        # Individual cleaned file path (optional)
        individual_output = None
        if save_individual:
            individual_output = pdf_file.with_name(pdf_file.stem + "_cleaned.txt")

        formatted_text = process_pdf(pdf_file, individual_output)
        all_cleaned_text += formatted_text

    # Save combined corpus
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(all_cleaned_text)

    print(f"\nFinal cleaned corpus saved to: {output_file}")


- Process single pdf

In [9]:
file_path = pdfs_path + "OS - 1.pdf" 
output_path = file_path[:-3] + "_cleaned.txt"
extract_images_from_pdf(file_path,"./images",250,250)

Extracted 43 images meeting size requirements from OS - 1.pdf into images
